# HƯỚNG DẪN TEST MULTI-RAG AGENT PIPELINES

## Các Pipeline Chính:
1. **Pipeline Ingestion**: Load → Chunk → Embed documents
2. **Pipeline Retrieval**: Search → Rerank → Query expansion
3. **Pipeline Agents**: Router → Domain RAG agents → Synthesizer
4. **Pipeline Full Graph**: Complete LangGraph workflow

---
## PIPELINE 1: INGESTION PIPELINE
Test quá trình load tài liệu, chia nhỏ và embedding

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

raw_paths = {
    "finance": PROJECT_ROOT / "data" / "raw" / "finance",
    "tech": PROJECT_ROOT / "data" / "raw" / "tech",
    "general": PROJECT_ROOT / "data" / "raw" / "general",
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Raw data paths:")
for domain, p in raw_paths.items():
    print(f"  {domain}: {p}")


PROJECT_ROOT: e:\AI Agent\ai-agent\multi-rag-agent
Raw data paths:
  finance: e:\AI Agent\ai-agent\multi-rag-agent\data\raw\finance
  tech: e:\AI Agent\ai-agent\multi-rag-agent\data\raw\tech
  general: e:\AI Agent\ai-agent\multi-rag-agent\data\raw\general


In [2]:
# TEST 1.1: Document Loading
from src.ingestion.loader import DocumentLoader
from config.settings import settings

loader = DocumentLoader()

# Test load từ mỗi domain
domains = ["finance", "tech", "general"]
loaded_docs = {}

print("=" * 80)
print("TEST 1.1: DOCUMENT LOADING")
print("=" * 80)

for domain in domains:
    path = getattr(settings, f"{domain}_data_path")
    docs = loader.load_from_directory(str(path))
    loaded_docs[domain] = docs
    
    print(f"\n{domain.upper()}:")
    print(f"  - Documents loaded: {len(docs)}")
    if docs:
        print(f"  - First doc source: {docs[0].metadata.get('source', 'N/A')}")
        print(f"  - Content length: {len(docs[0].page_content)} chars")
        print(f"  - Preview: {docs[0].page_content[:200]}...")

e:\AI Agent\ai-agent\multi-rag-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\AI Agent\ai-agent\multi-rag-agent\src\retrieval\vector_store.py:51: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.stores[domain] = Chroma(


TEST 1.1: DOCUMENT LOADING


0it [00:00, ?it/s]



FINANCE:
  - Documents loaded: 0


0it [00:00, ?it/s]



TECH:
  - Documents loaded: 0


0it [00:00, ?it/s]


GENERAL:
  - Documents loaded: 0


In [3]:
# TEST 1.2: Document Chunking
from src.ingestion.chunker import DocumentChunker

chunker = DocumentChunker(
    chunk_size=settings.chunk_size,
    chunk_overlap=settings.chunk_overlap
)

print("\n" + "=" * 80)
print("TEST 1.2: DOCUMENT CHUNKING")
print("=" * 80)

chunked_docs = {}

for domain, docs in loaded_docs.items():
    chunks = chunker.chunk(docs)
    chunked_docs[domain] = chunks
    
    print(f"\n{domain.upper()}:")
    print(f"  - Original docs: {len(docs)}")
    print(f"  - Chunks created: {len(chunks)}")
    print(f"  - Avg chunks per doc: {len(chunks)/len(docs) if docs else 0:.1f}")
    
    if chunks:
        print(f"  - Chunk size: {len(chunks[0].page_content)} chars")
        print(f"  - Preserved metadata: {list(chunks[0].metadata.keys())}")


TEST 1.2: DOCUMENT CHUNKING

FINANCE:
  - Original docs: 0
  - Chunks created: 0
  - Avg chunks per doc: 0.0

TECH:
  - Original docs: 0
  - Chunks created: 0
  - Avg chunks per doc: 0.0

GENERAL:
  - Original docs: 0
  - Chunks created: 0
  - Avg chunks per doc: 0.0


In [4]:
# TEST 1.3: Embedding Generation
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model=settings.embedding_model,
    google_api_key=settings.google_api_key
)

print("\n" + "=" * 80)
print("TEST 1.3: EMBEDDING GENERATION")
print("=" * 80)

# Test embedding on sample chunks
test_texts = [
    "What is Model Context Protocol?",
    "Explain a taxonomy of Agentic Systems",
    "Describe the flow of context management for agents"
]

for i, text in enumerate(test_texts, 1):
    vector = embeddings.embed_query(text)
    print(f"\nText {i}: {text}")
    print(f"  - Embedding dimension: {len(vector)}")
    print(f"  - Vector norm: {sum(v**2 for v in vector)**0.5:.4f}")
    print(f"  - First 3 values: {vector[:3]}")


TEST 1.3: EMBEDDING GENERATION

Text 1: What is Model Context Protocol?
  - Embedding dimension: 3072
  - Vector norm: 1.0000
  - First 3 values: [0.0072358977, 0.012594142, 0.010012589]

Text 2: Explain a taxonomy of Agentic Systems
  - Embedding dimension: 3072
  - Vector norm: 1.0000
  - First 3 values: [-0.013549116, -0.007166188, -0.012956386]

Text 3: Describe the flow of context management for agents
  - Embedding dimension: 3072
  - Vector norm: 1.0000
  - First 3 values: [-0.009046569, 0.007833544, 0.021646677]


In [5]:
# TEST 1.4: Vector Store Integration
from src.retrieval import vector_store_manager
import time

print("\n" + "=" * 80)
print("TEST 1.4: VECTOR STORE INTEGRATION")
print("=" * 80)

# Check current vector store status
print("\nVector Store Statistics BEFORE ingestion:")
for domain in domains:
    stats = vector_store_manager.get_store_stats(domain)
    print(f"  {domain}: {stats}")

# Add chunks to vector store (với batch processing để tránh rate limit)
BATCH_SIZE = 10
SLEEP_TIME = 10  # seconds

print("\nInserting chunks into vector stores...")
for domain in domains:
    if not chunked_docs[domain]:
        print(f"  {domain}: Skip (no chunks)")
        continue
    
    chunks = chunked_docs[domain]
    print(f"\n  {domain}: Adding {len(chunks)} chunks...")
    
    for i in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[i : i + BATCH_SIZE]
        vector_store_manager.add_documents(domain, batch)
        print(f"    - Added batch: {i + len(batch)} / {len(chunks)}")
        
        # Pause between batches
        if i + BATCH_SIZE < len(chunks):
            time.sleep(SLEEP_TIME)

# Check vector store after ingestion
print("\nVector Store Statistics AFTER ingestion:")
for domain in domains:
    stats = vector_store_manager.get_store_stats(domain)
    print(f"  {domain}: {stats}")


TEST 1.4: VECTOR STORE INTEGRATION

Vector Store Statistics BEFORE ingestion:
  finance: {'domain': 'finance', 'document_count': 0, 'embedding_dimension': 768, 'embedding_model': 'gemini-embedding-001', 'status': 'active'}
  tech: {'domain': 'tech', 'document_count': 630, 'embedding_dimension': 768, 'embedding_model': 'gemini-embedding-001', 'status': 'active'}
  general: {'domain': 'general', 'document_count': 0, 'embedding_dimension': 768, 'embedding_model': 'gemini-embedding-001', 'status': 'active'}

Inserting chunks into vector stores...
  finance: Skip (no chunks)
  tech: Skip (no chunks)
  general: Skip (no chunks)

Vector Store Statistics AFTER ingestion:
  finance: {'domain': 'finance', 'document_count': 0, 'embedding_dimension': 768, 'embedding_model': 'gemini-embedding-001', 'status': 'active'}
  tech: {'domain': 'tech', 'document_count': 630, 'embedding_dimension': 768, 'embedding_model': 'gemini-embedding-001', 'status': 'active'}
  general: {'domain': 'general', 'documen

---
## PIPELINE 2: RETRIEVAL PIPELINE
Test quá trình tìm kiếm, reranking, và query expansion

In [6]:
# TEST 2.1: Basic Vector Search
print("\n" + "=" * 80)
print("TEST 2.1: BASIC VECTOR SEARCH")
print("=" * 80)

test_queries = {
    "tech": "Explain a taxonomy of Agentic Systems",
    "finance": "What is investment portfolio diversification?",
    "general": "How does machine learning work?"
}

for domain, query in test_queries.items():
    print(f"\n{domain.upper()} Query: {query}")
    
    docs = vector_store_manager.search(
        domain=domain,
        query=query,
        k=3
    )
    
    print(f"  Retrieved: {len(docs)} documents")
    for i, doc in enumerate(docs, 1):
        print(f"\n  Doc {i}:")
        print(f"    Source: {doc.metadata.get('source', 'N/A')}")
        print(f"    Preview: {doc.page_content[:150]}...")


TEST 2.1: BASIC VECTOR SEARCH

TECH Query: Explain a taxonomy of Agentic Systems
  Retrieved: 3 documents

  Doc 1:
    Source: e:\AI Agent\ai-agent\multi-rag-agent\data\raw\tech\Day_1_v4.pdf
    Preview: the last.
 
 
 
 
Figure 2: Agentic system in 5 steps...

  Doc 2:
    Source: e:\AI Agent\ai-agent\multi-rag-agent\data\raw\tech\Day_1_v4.pdf
    Preview: Introduction to Agents and Agent architectures
November 2025
14
A Taxonomy of Agentic Systems
Understanding the 5-step operational loop is the first p...

  Doc 3:
    Source: e:\AI Agent\ai-agent\multi-rag-agent\data\raw\tech\Day_1_v4.pdf
    Preview: Model (the "Brain"), the actionable Tools (the "Hands"), and the governing Orchestration 
Layer (the "Nervous System"). It is the seamless integration...

FINANCE Query: What is investment portfolio diversification?
  Retrieved: 0 documents

GENERAL Query: How does machine learning work?
  Retrieved: 0 documents


In [ ]:
# TEST 2.2: Query Expansion
from src.retrieval.query_expansion import QueryExpander

print("\n" + "=" * 80)
print("TEST 2.2: QUERY EXPANSION")
print("=" * 80)

expander = QueryExpander()
test_query = "Explain a taxonomy of Agentic Systems"

print(f"\nOriginal query: {test_query}")
expanded = expander.get_expanded_queries(test_query)

print(f"\nExpanded queries: {len(expanded)}")
for i, q in enumerate(expanded, 1):
    print(f"  {i}. {q}")


TEST 2.2: QUERY EXPANSION

Original query: Explain a taxonomy of Agentic Systems


LLM quota exhausted during query expansion: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 59.384704048s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini AP


Expanded queries: 1
  1. Explain a taxonomy of Agentic Systems


In [10]:
# TEST 2.3: Reranking
from src.retrieval.reranker import Reranker

print("\n" + "=" * 80)
print("TEST 2.3: DOCUMENT RERANKING")
print("=" * 80)

reranker = Reranker()
domain = "tech"
query = "How do LLMs work?"

# Get documents
retrieved_docs = vector_store_manager.search(
    domain=domain,
    query=query,
    k=5
)

print(f"\nQuery: {query}")
print(f"Retrieved: {len(retrieved_docs)} documents")

# Before reranking
print("\nBefore reranking:")
for i, doc in enumerate(retrieved_docs[:3], 1):
    print(f"  {i}. {doc.page_content[:100]}...")

# Rerank
reranked = reranker.rerank(query, retrieved_docs, k=3)

print(f"\nAfter reranking (top 3):")
for i, (doc, score) in enumerate(reranked, 1):
    print(f"  {i}. Score: {score:.4f}")
    print(f"     {doc.page_content[:100]}...")


TEST 2.3: DOCUMENT RERANKING

Query: How do LLMs work?
Retrieved: 5 documents

Before reranking:
  1. reasoning model, actionable tools, and a governing orchestration layer. A short version is 
“LMs in ...
  2. deleted. If it is augmented, it may be updated.
Second, an LLM is presented with both the existing m...
  3. new one.
• DELETE / INVALIDATE: If the new information makes an old memory completely irrelevant 
or...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3569.76it/s]



After reranking (top 3):
  1. Score: 5.2437
     deleted. If it is augmented, it may be updated.
Second, an LLM is presented with both the existing m...
  2. Score: 2.7067
     meaningful insights, functioning. Think of it as an LLM-driven ETL (Extract, Transform, 
Load) pipel...
  3. Score: 2.5112
     conversation history, to the memory manager.
2. Extraction & Filtering: The memory manager uses an L...


---
## PIPELINE 3: INDIVIDUAL AGENTS
Test từng agent riêng lẻ trước khi chạy full graph

In [11]:
# TEST 3.1: Router Agent
from src.agents.router_agent import RouterAgent

print("\n" + "=" * 80)
print("TEST 3.1: ROUTER AGENT")
print("=" * 80)

router = RouterAgent()

test_queries = [
    "What is Model Context Protocol?",
    "Explain a taxonomy of Agentic Systems",
    "Describe the flow of context management for agents"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    result = router.run({"query": query})
    
    print(f"  Route: {result.get('route')}")
    print(f"  Active agents: {result.get('agents')}")
    print(f"  Confidence: {result.get('confidence', 0):.2f}")
    print(f"  Reasoning: {result.get('reasoning', '')[:100]}...")


TEST 3.1: ROUTER AGENT

Query: What is Model Context Protocol?


Error invoking LLM: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 9.752416596s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https:

  Route: general
  Active agents: ['general']
  Confidence: 0.50
  Reasoning: Fallback classification due to error...

Query: Explain a taxonomy of Agentic Systems


Error invoking LLM: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 35.611842431s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https

  Route: general
  Active agents: ['general']
  Confidence: 0.50
  Reasoning: Fallback classification due to error...

Query: Describe the flow of context management for agents


Error invoking LLM: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 1.602464439s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https:

  Route: general
  Active agents: ['general']
  Confidence: 0.50
  Reasoning: Fallback classification due to error...


In [ ]:
# TEST 3.2: Domain RAG Agents
from src.agents.rag_agent import RAGAgent

print("\n" + "=" * 80)
print("TEST 3.2: DOMAIN RAG AGENTS")
print("=" * 80)

domain_agents = {
    "tech": RAGAgent("tech"),
    "finance": RAGAgent("finance"),
    "general": RAGAgent("general")
}

test_queries_by_domain = {
    "tech": "What is transformer architecture?",
    "finance": "Explain portfolio diversification strategy",
    "general": "How does photosynthesis work?"
}

for domain, agent in domain_agents.items():
    query = test_queries_by_domain[domain]
    print(f"\n{domain.upper()} Agent:")
    print(f"  Query: {query}")
    
    result = agent.run({"query": query})
    
    print(f"  Success: {result.get('success')}")
    print(f"  Retrieved docs: {len(result.get('retrieved_docs', []))}")
    print(f"\n  Answer (first 300 chars):")
    print(f"  {result.get('answer', '')[:300]}...")
    print(f"\n  Sources:")
    for source in result.get('sources', [])[:2]:
        print(f"    - {source}")

In [ ]:
# TEST 3.3: Supervisor Agent (Multi-Agent Coordination)
from src.agents.supervisor_agent import SupervisorAgent

print("\n" + "=" * 80)
print("TEST 3.3: SUPERVISOR AGENT (Multi-Agent Coordination)")
print("=" * 80)

supervisor = SupervisorAgent()

# Test complex query requiring multiple agents
complex_query = "Compare AI applications in finance and technology sectors"

print(f"\nComplex Query: {complex_query}")
print("  Using agents: ['finance', 'tech']")

result = supervisor.run({
    "query": complex_query,
    "active_agents": ["finance", "tech"]
})

print(f"\nSuccess: {result.get('success')}")
print(f"\nAgent Responses:")
for agent_name, response in result.get('responses', {}).items():
    print(f"\n  {agent_name}:")
    print(f"    Success: {response.get('success')}")
    print(f"    Answer (first 200 chars): {response.get('answer', '')[:200]}...")

In [ ]:
# TEST 3.4: Synthesizer Agent
from src.agents.synthesizer_agent import SynthesizerAgent

print("\n" + "=" * 80)
print("TEST 3.4: SYNTHESIZER AGENT")
print("=" * 80)

synthesizer = SynthesizerAgent()

# Create sample multi-agent responses
sample_responses = {
    "tech": {
        "answer": "AI is transforming technology with transformer models and LLMs.",
        "sources": ["doc1.pdf", "doc2.pdf"]
    },
    "finance": {
        "answer": "AI is revolutionizing finance through algorithmic trading and risk assessment.",
        "sources": ["doc3.pdf", "doc4.pdf"]
    }
}

print(f"\nInput responses from {len(sample_responses)} agents")
result = synthesizer.run({
    "query": "How is AI transforming different sectors?",
    "agent_responses": sample_responses
})

print(f"\nSynthesized Answer:")
print(f"  {result.get('final_answer', '')[:400]}...")
print(f"\nKey Themes: {result.get('key_themes', [])}")
print(f"All Sources: {result.get('all_sources', [])}")

---
## PIPELINE 4: FULL GRAPH WORKFLOW
Test complete LangGraph workflow từ đầu đến cuối

In [ ]:
# TEST 4.1: Build and Initialize Graph
from src.graph.workflow import build_agent_graph

print("\n" + "=" * 80)
print("TEST 4.1: BUILD LANGGRAPH WORKFLOW")
print("=" * 80)

try:
    graph = build_agent_graph()
    compiled_graph = graph.compile()
    
    print("✓ Graph built successfully")
    print(f"✓ Graph compiled")
    print(f"  Entry point: {graph.entry_point}")
    
except Exception as e:
    print(f"✗ Error building graph: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# TEST 4.2: Single Query Through Full Pipeline
from langchain_core.messages import HumanMessage
import time

print("\n" + "=" * 80)
print("TEST 4.2: SINGLE DOMAIN QUERY (Tech)")
print("=" * 80)

# Initialize state
initial_query = "How do language models understand context?"

initial_state = {
    "messages": [HumanMessage(content=initial_query)],
    "query": initial_query,
    "route": "",
    "active_agents": [],
    "router_confidence": 0.0,
    "retrieved_docs": {},
    "reranked_docs": {},
    "agent_responses": {},
    "final_answer": "",
    "key_themes": [],
    "sources": [],
    "iteration": 0,
    "max_iterations": 5,
    "error": "",
    "status": "pending",
    "start_time": time.time(),
    "processing_times": {}
}

print(f"\nQuery: {initial_query}")
print("\nRunning through graph...")

try:
    result = compiled_graph.invoke(initial_state)
    
    print("\n✓ Query completed successfully")
    print(f"\nRoute: {result.get('route')}")
    print(f"Active Agents: {result.get('active_agents')}")
    print(f"Status: {result.get('status')}")
    
    print(f"\nFinal Answer (first 500 chars):")
    print(f"  {result.get('final_answer', '')[:500]}...")
    
    print(f"\nKey Themes: {result.get('key_themes', [])}")
    print(f"Sources: {result.get('sources', [])[:3]}")
    
    # Processing times
    if result.get('processing_times'):
        print(f"\nProcessing Times:")
        for component, time_taken in result['processing_times'].items():
            print(f"  {component}: {time_taken:.2f}s")
    
except Exception as e:
    print(f"\n✗ Error during graph execution: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# TEST 4.3: Multi-Domain Query
print("\n" + "=" * 80)
print("TEST 4.3: MULTI-DOMAIN QUERY")
print("=" * 80)

multi_query = "How are AI and machine learning impacting finance and technology sectors?"

initial_state_multi = {
    "messages": [HumanMessage(content=multi_query)],
    "query": multi_query,
    "route": "",
    "active_agents": [],
    "router_confidence": 0.0,
    "retrieved_docs": {},
    "reranked_docs": {},
    "agent_responses": {},
    "final_answer": "",
    "key_themes": [],
    "sources": [],
    "iteration": 0,
    "max_iterations": 5,
    "error": "",
    "status": "pending",
    "start_time": time.time(),
    "processing_times": {}
}

print(f"\nQuery: {multi_query}")
print("(This query should trigger multiple agents)")

try:
    result_multi = compiled_graph.invoke(initial_state_multi)
    
    print("\n✓ Multi-domain query completed")
    print(f"\nRoute: {result_multi.get('route')}")
    print(f"Active Agents: {result_multi.get('active_agents')}")
    
    print(f"\nFinal Answer (first 500 chars):")
    print(f"  {result_multi.get('final_answer', '')[:500]}...")
    
    print(f"\nAgent Responses Received:")
    for agent_name in result_multi.get('agent_responses', {}).keys():
        print(f"  ✓ {agent_name}")
    
except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# TEST 4.4: Batch Testing Multiple Queries
print("\n" + "=" * 80)
print("TEST 4.4: BATCH TESTING (5 Different Queries)")
print("=" * 80)

test_queries_batch = [
    "What are transformer architectures?",
    "Explain stock market indices",
    "How does cloud computing work?",
    "Compare Python vs JavaScript",
    "What is quantum computing?"
]

results_summary = []

for i, query in enumerate(test_queries_batch, 1):
    print(f"\n[{i}/5] Query: {query}")
    
    initial_state_batch = {
        "messages": [HumanMessage(content=query)],
        "query": query,
        "route": "",
        "active_agents": [],
        "router_confidence": 0.0,
        "retrieved_docs": {},
        "reranked_docs": {},
        "agent_responses": {},
        "final_answer": "",
        "key_themes": [],
        "sources": [],
        "iteration": 0,
        "max_iterations": 5,
        "error": "",
        "status": "pending",
        "start_time": time.time(),
        "processing_times": {}
    }
    
    try:
        result_batch = compiled_graph.invoke(initial_state_batch)
        
        results_summary.append({
            "query": query,
            "status": "✓ Success",
            "route": result_batch.get('route'),
            "agents": result_batch.get('active_agents', [])
        })
        
        print(f"  Route: {result_batch.get('route')}")
        print(f"  Agents: {result_batch.get('active_agents', [])}")
        
    except Exception as e:
        results_summary.append({
            "query": query,
            "status": f"✗ Error: {str(e)[:50]}",
            "route": "N/A",
            "agents": []
        })
        print(f"  ✗ Error: {e}")

print("\n" + "=" * 80)
print("BATCH TEST SUMMARY")
print("=" * 80)
for item in results_summary:
    print(f"\n{item['status']}: {item['query'][:40]}...")
    print(f"  Route: {item['route']}, Agents: {item['agents']}")

---
## PIPELINE 5: API ENDPOINT TESTING (Optional)
Test qua API endpoints nếu cần chạy ở server mode

In [ ]:
# TEST 5: (Optional) Test API Endpoints
print("\n" + "=" * 80)
print("TEST 5: API ENDPOINT TESTING (Optional)")
print("=" * 80)

print("""
Để test API endpoints, chạy các lệnh sau trong terminal:

1. Start server:
   cd e:/AI Agent/ai-agent/multi-rag-agent
   python -m uvicorn api.main:app --reload --port 8000

2. In another terminal, test endpoints:
   
   # Health check
   curl http://localhost:8000/health
   
   # Query endpoint
   curl -X POST http://localhost:8000/api/query \\
     -H "Content-Type: application/json" \\
     -d '{"query": "What is MCP?", "domains": ["tech"]}'
   
   # Router endpoint
   curl -X POST http://localhost:8000/api/route \\
     -H "Content-Type: application/json" \\
     -d '{"query": "What is stock market analysis?"}'
   
   # Retrieval endpoint
   curl -X POST http://localhost:8000/api/retrieve \\
     -H "Content-Type: application/json" \\
     -d '{"query": "Machine learning", "domain": "tech", "k": 5}'

3. Or use Python requests:
""")

print("Example Python code:")
print("""
import requests
import json

BASE_URL = "http://localhost:8000"

# Query endpoint
response = requests.post(
    f"{BASE_URL}/api/query",
    json={"query": "What is MCP?", "domains": ["tech"]}
)
print(response.json())

# Route endpoint
response = requests.post(
    f"{BASE_URL}/api/route",
    json={"query": "Stock market analysis"}
)
print(response.json())
""")

---
## SUMMARY & NEXT STEPS

### Thứ tự Test Được Khuyến Nghị:
1. **Pipeline 1 (Ingestion)**: Đảm bảo dữ liệu được load, chunk, embed đúng
2. **Pipeline 2 (Retrieval)**: Kiểm tra search, reranking, query expansion hoạt động
3. **Pipeline 3 (Agents)**: Test từng agent riêng lẻ
4. **Pipeline 4 (Graph)**: Chạy full workflow tích hợp
5. **Pipeline 5 (API)**: Nếu cần deploy qua server

### Các Metrics Cần Monitor:
- ✓ Success rate của mỗi pipeline
- ⏱ Processing time từng component
- 📊 Document retrieval quality
- 🎯 Router accuracy (routing to correct domains)
- 📝 Answer quality và relevance

### Troubleshooting:
- Check logs in `logs/` folder
- Rate limiting issues → Adjust BATCH_SIZE and SLEEP_TIME
- API key issues → Check `settings.py`
- Vector store issues → Rebuild với `vector_store_manager.reset()`